In [0]:
# Databricks notebook source
# MAGIC %md
# MAGIC # Task 3 — Load
# MAGIC Takes this run's silver snapshot and publishes it two ways:
# MAGIC
# MAGIC 1. **Gold** — a Delta table in the lakehouse, plus a dated CSV in the gold `exports` volume
# MAGIC 2. **Database** — the Neon Postgres table
# MAGIC
# MAGIC **Re-runs:** the gold table and the CSV are replaced, and the Postgres rows for this
# MAGIC `run_date` are deleted before the insert, so nothing is duplicated. If the columns no longer
# MAGIC match the existing Postgres table, the task stops and explains why; set `allow_schema_change`
# MAGIC to `true` to rebuild that table.

# COMMAND ----------

# MAGIC %pip install sqlalchemy psycopg2-binary --quiet

# COMMAND ----------

# ============================================================
# 1. PARAMETERS
# ============================================================
%pip install sqlalchemy
dbutils.widgets.text("catalog", "gdp_etl_catalog")
dbutils.widgets.text("run_date", "")                  # set by the job; blank = today (UTC)
dbutils.widgets.text("silver_table", "countries_gdp")
dbutils.widgets.text("gold_table", "countries_gdp")
dbutils.widgets.text("pg_host", "ep-twilight-sound-ayi2ejh4.c-5.us-east-2.aws.neon.tech")
dbutils.widgets.text("pg_database", "GDP_ETL_DB")
dbutils.widgets.text("pg_user", "neondb_owner")
dbutils.widgets.text("pg_table", "countries_by_gdp")
dbutils.widgets.text("secret_schema", "bronze")       # schema holding the 'neon_password' secret
dbutils.widgets.text("allow_schema_change", "false")  # true = rebuild the Postgres table if columns differ

CATALOG             = dbutils.widgets.get("catalog")
SILVER_TABLE        = f"{CATALOG}.silver.{dbutils.widgets.get('silver_table')}"
GOLD_TABLE          = f"{CATALOG}.gold.{dbutils.widgets.get('gold_table')}"
EXPORT_DIR          = f"/Volumes/{CATALOG}/gold/exports"
BRONZE_DIR          = f"/Volumes/{CATALOG}/bronze/raw_html"

PG_HOST             = dbutils.widgets.get("pg_host")
PG_DATABASE         = dbutils.widgets.get("pg_database")
PG_USER             = dbutils.widgets.get("pg_user")
PG_TABLE            = dbutils.widgets.get("pg_table")
SECRET_SCHEMA       = dbutils.widgets.get("secret_schema")
ALLOW_SCHEMA_CHANGE = dbutils.widgets.get("allow_schema_change").strip().lower() == "true"

# COMMAND ----------

# ============================================================
# 2. IMPORTS
# ============================================================

import os
from datetime import date, datetime, timezone
from urllib.parse import quote_plus

import pandas as pd
from sqlalchemy import create_engine, text

RUN_DATE = date.fromisoformat(dbutils.widgets.get("run_date")) if dbutils.widgets.get("run_date") \
    else datetime.now(timezone.utc).date()

STAGE = "load"

# COMMAND ----------

# ============================================================
# 3. FUNCTIONS
# ============================================================

LOG_LINES = []


def remove_if_exists(path):
    """Volumes don't allow overwriting a file in place, so a re-run deletes it first."""
    if os.path.exists(path):
        os.remove(path)


def log_progress(message):
    """Collect a timestamped message and print it. Returns nothing."""
    line = f"{datetime.now(timezone.utc):%Y-%m-%d %H:%M:%S} : [{STAGE}] {message}"
    LOG_LINES.append(line)
    print(line)


def write_log(log_dir, run_date, stage):
    """Save this task's collected log lines as one file."""
    path = f"{log_dir}/etl_log_{run_date}_{stage}.txt"
    remove_if_exists(path)
    with open(path, "w", encoding="utf-8") as f:
        f.write("\n".join(LOG_LINES) + "\n")
    print(f"Log written to {path}")


def load_to_db(df, engine, table, run_date, allow_schema_change):
    """Load the snapshot into Postgres and return what the write did.

    Deleting this run_date's rows first is what makes a re-run safe. If the scraped
    columns no longer match the existing table, an append would fail, so the task
    either stops with an explanation or drops and rebuilds the table on request.
    """
    outcome = "created"
    with engine.begin() as conn:
        exists = conn.execute(text("SELECT to_regclass(:t)"), {"t": table}).scalar()
        if exists:
            existing = [
                row[0] for row in conn.execute(
                    text("SELECT column_name FROM information_schema.columns "
                         "WHERE table_name = :t ORDER BY ordinal_position"),
                    {"t": table},
                )
            ]
            if existing != list(df.columns):
                if not allow_schema_change:
                    raise ValueError(
                        "Column mismatch with the existing Postgres table. Set allow_schema_change=true "
                        "to drop and rebuild it, or point pg_table at a new name.\n"
                        f"  existing : {existing}\n"
                        f"  new      : {list(df.columns)}"
                    )
                conn.execute(text(f'DROP TABLE "{table}"'))
                outcome = "rebuilt (previous snapshots dropped)"
            else:
                conn.execute(text(f'DELETE FROM "{table}" WHERE run_date = :d'), {"d": run_date})
                outcome = "this run_date replaced"

    df.to_sql(table, engine, if_exists="append", index=False)
    return outcome

# COMMAND ----------

# ============================================================
# 4. TASK EXECUTION
# ============================================================

log_progress(f"Initiating load (run_date {RUN_DATE})")

# --- Gold: this run's snapshot, kept in the lakehouse ---------------
# CREATE OR REPLACE rebuilds the table each run, so re-running is safe by construction.
spark.sql(f"""
    CREATE OR REPLACE TABLE {GOLD_TABLE} AS
    SELECT * FROM {SILVER_TABLE} WHERE run_date = '{RUN_DATE}'
""")
df = spark.table(GOLD_TABLE).toPandas()
if df.empty:
    raise ValueError(f"No rows in {SILVER_TABLE} for run_date {RUN_DATE}")
log_progress(f"Gold table refreshed: {GOLD_TABLE} ({len(df)} rows)")

# --- Gold: the shareable file ---------------------------------------
currency = df["currency"].iloc[0]
csv_path = f"{EXPORT_DIR}/gdp_{RUN_DATE}_{currency}.csv"
remove_if_exists(csv_path)
df.to_csv(csv_path, index=False)
log_progress(f"CSV exported to {csv_path}")

# --- Database: Neon Postgres ----------------------------------------
password = dbutils.secrets.get(catalog=CATALOG, schema=SECRET_SCHEMA, key="neon_password")
engine = create_engine(
    f"postgresql+psycopg2://{PG_USER}:{quote_plus(password)}@{PG_HOST}/{PG_DATABASE}?sslmode=require"
)
log_progress(f"Database connection initiated ({PG_HOST}/{PG_DATABASE})")

outcome = load_to_db(df, engine, PG_TABLE, RUN_DATE, ALLOW_SCHEMA_CHANGE)
log_progress(f"Data loaded into {PG_TABLE} ({outcome})")

check = pd.read_sql(
    f'SELECT COUNT(*) AS rows, COUNT(DISTINCT run_date) AS snapshots FROM "{PG_TABLE}"', engine
)
print(check)
log_progress(f"Verification: {int(check.iloc[0, 0])} rows across {int(check.iloc[0, 1])} snapshot(s)")

engine.dispose()
log_progress("Database connection closed")
log_progress("Load complete")
write_log(BRONZE_DIR, RUN_DATE, STAGE)